# Week 9 lab · Brixton Road: still London's worst road

**LDSCI6253 Information Presentation & Data Visualisation** · Northeastern University London

In the lecture you watched one chart of Marylebone Road grow into a story, a move at a time: the finding in the
title, notes where it happened, one line in colour and the rest grey, then a martini glass. Today you use **the same
moves, and two new ones**, on a road whose story is harder to tell.

**Brixton Road** (Lambeth, kerbside) had 174 µg/m³ of NO₂ in 2010, the worst of London's 47
roadside monitors. By 2025 it was down to 48. Good news? It is still the worst of the 39, and
the only one over the legal limit of 40. A chart of Brixton Road alone shows the fall but hides the second half.
**You have to isolate it from every other road** to tell the whole story.

| Part | You will… | Advanced move | Time |
|---|---|---|---|
| **A** | find the numbers the story rests on | filtering, isolating one road | 8 min |
| **B** | fix a chart that tells the fall wrong | the right channel type for time | 5 min |
| **C** | say the finding, and mark the limit | layered marks: `mark_rect`, `mark_rule`, `mark_text` | 12 min |
| **D** | Brixton in red against every road in grey | a conditional channel: `alt.when` | 15 min |
| **E** | lead first, then let the reader explore | a parameter and chapters (martini glass) | 12 min |
| **F** | hand in: your chart and one sentence | | 3 min |

**How to work, in pairs.** One of you types, the other checks. Before each Run, the checker says what they expect
to see; after it, compare. Swap at each letter. The code from the lecture slides is in `week9_build_its.py`: keep
it open, every part says which Build-it to look at.

- Where you see `___`, replace it with code, then run the cell (Shift + Enter). Run cells **in order**.
- A ✅ or ❌ line after a cell says whether you are on track.
- Stuck? Open the 💡 **Hint**, then the Build-it named.

**If you get an error**
- `NameError`: a cell above was not run (run the cells above, in order), or a name is missing its quotes.
- An error that mentions **`BLANK`** or `___`: a blank is still empty.
- `SchemaValidationError`: a column name is misspelt, or missing its type (`'year:Q'`).
- A chart that is empty: a filter keeps nothing. Check the spelling inside the quotes (`'LB4'`).

## 0 · Set up

Run this cell first. It loads the data the lecture used: one row per monitor per year.

In [ ]:
import altair as alt
import pandas as pd
from pathlib import Path

LIVE = "https://slideforge-rpjj.onrender.com/assets/lesson/ipdv/week9/"

def load(name):
    """Read name.csv from beside this notebook, or from the course site if it is not here."""
    here = Path(name + ".csv")
    return pd.read_csv(here if here.exists() else LIVE + name + ".csv")

def check(ok, good, hint):
    """Print ✅ and what you found, or ❌ and what to look at."""
    print('✅ ' + good if ok else '❌ Not yet: ' + hint)

___ = 'BLANK: replace ___ with your code'   # an empty blank names itself in the error

air = load('no2_london')                                # code, site, borough, type, year, no2
road = air[air['type'].isin(['Roadside', 'Kerbside'])]  # monitors beside a road
road.head()

## A · Isolate the road (8 min · Build-it *Start with the bare line*)

A story rests on a few numbers. Find them. Brixton Road's monitor code is `'LB4'`.

**A1.** Keep only Brixton Road's rows, the way the lecture kept Marylebone Road's (`my1`).

In [ ]:
brixton = road[road['code'] == ___]
brixton[['year', 'no2']]

In [ ]:
check(len(brixton) == 12, 'Brixton Road has 12 years of data.',
      "keep the rows whose code is 'LB4'.")

**A2.** The two halves of the story. Work out the fall since 2010, and how Brixton Road ranks among the
roadside monitors in 2025.

In [ ]:
first = brixton['no2'].iloc[0]          # 2010
last = brixton['no2'].iloc[-1]          # 2025
fall = round((first - last) / ___ * 100)

latest = road[road['year'] == 2025]
worse = (latest['no2'] > last).sum()    # how many roads were worse than Brixton
over = (latest['no2'] > ___).sum()     # how many roads were over the legal limit
print(f'Fell {fall}%. In 2025: {worse} roads worse, {over} of {len(latest)} over the limit.')

In [ ]:
check(fall == 72 and worse == 0 and over == 1,
      'Fell 72%, and still the worst: no road worse, and the only one over the limit.',
      'the fall divides by the first year; the limit is 40.')

<details><summary>💡 <b>Hint</b></summary>

The fall is the drop divided by where it started: `(first - last) / first * 100`.
</details>

✏️ *In one sentence, the two halves of Brixton Road's story: …*

## B · An honest axis (5 min · lecture trick 1, *the window*)

This chart is the lecture's bare line on Brixton Road. Run it, and look at the years along the bottom.

In [ ]:
alt.Chart(brixton).mark_line(point=True).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
).properties(width=560, height=300)

**B1.** `year:O` treats years as labels in a row, so missing years vanish. Make the year a **quantity**, so each year
sits at its true distance, gaps and all. (`format='d'` shows 2010, not 2,010.)

In [ ]:
alt.Chart(brixton).mark_line(point=True).encode(
    x=alt.X('year:___', title=None, axis=alt.Axis(format='d'), scale=alt.Scale(nice=False)),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
).properties(width=560, height=300)

✏️ *Why does `year:O` make the fall look faster? …*

## C · Say it, and mark the limit (12 min · Build-its *Say the finding in the title*, *Write on the chart*)

Now build the chart in layers, as the lecture did with `line + limit`: each layer is its own `alt.Chart`, and `+`
puts them on top of each other. The one added **last** is drawn **on top**.

**C1.** A shaded band from the limit (40) to the top of the chart: "over the limit". `mark_rect` with `y` and `y2`
draws a band between two values.

In [ ]:
red = '#c8102e'
years = alt.X('year:Q', title=None, axis=alt.Axis(format='d'), scale=alt.Scale(domain=[2010, 2025]))
height = alt.Scale(domain=[0, 180])          # the axis stays the same in every chart from here

dots = alt.OverlayMarkDef(color=red)
line = alt.Chart(brixton).mark_line(point=dots, color=red, strokeWidth=3).encode(
    x=years, y=alt.Y('no2:Q', title='NO₂ (µg/m³)', scale=height))

band = alt.Chart().mark____(color=red, opacity=0.07).encode(
    y=alt.datum(___), y2=alt.datum(180))     # from the limit to the top
limit = alt.Chart().mark_rule(strokeDash=[6, 4]).encode(y=alt.datum(40))
limit_words = alt.Chart().mark_text(align='left', dy=-8).encode(
    x=alt.value(6), y=alt.datum(40), text=alt.value('Legal limit: 40'))   # 6 pixels in from the left

(band + limit + limit_words + line).properties(width=560, height=300)

<details><summary>💡 <b>Hint</b></summary>

`mark_rect` draws a rectangle. Give it where the band starts (`y`) and where it ends (`y2`).
</details>

**C2.** The title says the finding: **both halves**, in one sentence. The subtitle says how to read the chart.

In [ ]:
finding = ___
how = 'NO₂ yearly average at the kerb, µg/m³. Shaded: over the legal limit of 40. 2025 is provisional.'

(band + limit + limit_words + line).properties(
    width=560, height=300, title=alt.Title(finding, subtitle=how))

In [ ]:
check('worst' in finding.lower() or 'over' in finding.lower() or 'still' in finding.lower(),
      'The title says the second half too.',
      'a title that only says it fell hides that it is still the worst.')

## D · Brixton against every road (15 min · Build-it *One line in colour, the rest grey*)

"Still the worst" needs the other roads on the chart. The lecture drew them as a second, grey layer. Here is a
neater way: **one chart of every road, with a condition on its channels**. `alt.when(test).then(a).otherwise(b)`
gives the rows that pass the test one value, and the rest another.

**D1.** Colour Brixton Road red and every other road grey, thick and thin.

In [ ]:
is_brixton = alt.datum.code == ___

roads = alt.Chart(road).mark_line().encode(
    x=years,
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)', scale=height),
    detail='code:N',                                            # one line per monitor
    color=alt.when(is_brixton).then(alt.value(red)).otherwise(alt.value('#b8b8b8')),
    strokeWidth=alt.when(is_brixton).then(alt.value(3)).otherwise(alt.value(___)),
)
(band + limit + roads + limit_words).properties(width=560, height=300, title=alt.Title(finding, subtitle=how))

<details><summary>💡 <b>Hint</b></summary>

`alt.datum.code` is each row's `code`. The test is true for Brixton Road's rows only.
</details>

**D2.** No legend: label the lines where they end. Add the London median (the middle road each year) as a dashed
line, and a label at the end of Brixton's line and of the median's. `transform_aggregate` works out the median for
each year; `transform_filter` keeps only the last year for the labels.

In [ ]:
median = alt.Chart(road).transform_aggregate(
    no2='___(no2)', groupby=['year']
).mark_line(color='#555555', strokeDash=[2, 3]).encode(x=years, y=alt.Y('no2:Q', scale=height))

end = 2025
brixton_label = alt.Chart(brixton).transform_filter(alt.datum.year == end).mark_text(
    align='left', dx=8, color=red, fontWeight='bold').encode(
    x=years, y=alt.Y('no2:Q', scale=height), text=alt.value('Brixton Road'))
median_label = median.transform_filter(alt.datum.year == end).mark_text(
    align='left', dx=8, color='#555555').encode(text=alt.value('London median'))

story = band + limit + roads + median + limit_words + brixton_label + median_label
story.properties(width=560, height=300, title=alt.Title(finding, subtitle=how))

## E · Lead first, then let go (12 min · Build-it *A martini glass: chapters, then open*)

A martini glass tells the finding first, in a fixed order (the stem), then opens up for the reader (the bowl).
A **parameter** bound to buttons is the chapter; `transform_filter(chapter >= n)` shows a layer from chapter *n* on.

**E1.** Three chapters: 1. Brixton alone and the limit; 2. the notes; 3. every road and the median.

In [ ]:
chapter = alt.param(value=1, bind=alt.binding_radio(options=[1, 2, 3], name='Chapter '))

notes = pd.DataFrame({
    'year': [2019, 2020, 2021, 2023],
    'at': [172, 157, 142, 127],                                   # how high each note sits
    'text': ['Central London ULEZ', 'Lockdown', 'ULEZ widened', 'London-wide'],
})
rules = alt.Chart(notes).mark_rule(color='grey').encode(x=years, y=alt.Y('at:Q', scale=height), y2=alt.datum(0))
words = alt.Chart(notes).mark_text(align='left', dx=5).encode(x=years, y=alt.Y('at:Q', scale=height), text='text:N')

# each layer brings its own data, so each is filtered on its own
stem = rules.transform_filter(chapter >= ___) + words.transform_filter(chapter >= 2)
bowl = (roads.transform_filter(chapter >= ___) + median.transform_filter(chapter >= 3)
        + median_label.transform_filter(chapter >= 3))

(band + limit + bowl + limit_words + stem + line + brixton_label).add_params(___).properties(
    width=560, height=300, title=alt.Title(finding, subtitle=how))

<details><summary>💡 <b>Hint</b></summary>

A layer from chapter 2 on is filtered with `chapter >= 2`. A parameter only works once the chart is told about it: `.add_params(chapter)`.
</details>

**Stretch (if you finish).** Let the reader isolate *any* road in chapter 3: a hover selection on `code`, and make
the hovered road dark. Start from `hover = alt.selection_point(on='pointerover', fields=['code'], empty=False)` and
add it to `roads` with `.add_params(hover)`.

## F · Hand in (3 min)

Save the notebook. On Canvas, by midday on the Friday after the lab, hand in:

1. your chart from **E**, all three chapters working;
2. one sentence: **what your chart tells, and what a reader would need to disagree with it**.

✏️ *My chart tells … A reader who disagreed would need …*